# Find and define unique peptides for all isoforms
In this notebook we will use the results from the tryptic in silico digestion and try to define a unique peptide for every isofofrm in our proteome.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

In [ ]:
# Data directory for fasta data
mapping_dir = "../01_UniProt/data/mapping/"
digestion_output_dir = "data/digestion_output"

## Read in Dataframes

In [ ]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_full_proteome_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_full_proteome_filtered.csv'))
digestion_m20_full_proteome_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_m20_full_proteome_filtered.csv'))

# Isoform-specific peptides
Implementation uses set substraction, which is a fast and accurate method to filter. I does not account for isoforms for which we can't find a unique peptide.

In [ ]:
# 1. Separate your digestion into two buckets
# Canonical IDs (already stripped of -1) vs Isoform IDs (usually -2, -3, etc.)
canonical_ids = iso_canonical_mapping["Canonical_Isoform_ID"].dropna().tolist()
canonical_base_ids = [cid[:-2] if cid.endswith('-1') else cid for cid in canonical_ids]

canonical_peptides = set(digestion_full_proteome_filtered[
    digestion_full_proteome_filtered['ID'].isin(canonical_base_ids)
]['seq'])

isoform_df = digestion_full_proteome_filtered[
    ~digestion_full_proteome_filtered['ID'].isin(canonical_base_ids)
].copy()

# 2. Identify peptides in isoforms that are NEVER found in any canonical protein
all_isoform_peptides = set(isoform_df['seq'])
isoform_specific_peptides = all_isoform_peptides - canonical_peptides

# 3. Filter the isoform dataframe to keep only these specific identifiers
isoform_unique_db = isoform_df[isoform_df['seq'].isin(isoform_specific_peptides)].copy()

# 4. Optional: Ensure the peptide is unique to ONLY ONE isoform 
# (In case two isoforms share the same mutation)
final_counts = isoform_unique_db.groupby('seq')['ID'].nunique()
truly_unique_seqs = final_counts[final_counts == 1].index

isoform_unique_db = isoform_unique_db[isoform_unique_db['seq'].isin(truly_unique_seqs)]

In [ ]:
isoform_unique_db

In [ ]:
# Create a mapping for your database
lookup_table = isoform_unique_db[['ID', 'seq', 'len']].drop_duplicates()

# Sort by length if you prefer longer peptides for better MS/MS confidence
lookup_table = lookup_table.sort_values(['ID', 'len'], ascending=[True, False])

In [ ]:
lookup_table

# Slightly different implementaton to flag isoforms for which we can't find a unique peptide like this

Insertions/Alternative Exons: These almost always produce unique peptides.
Deletions: These create a new "junction" peptide (where two parts of the protein are joined that aren't joined in the canonical).
Truncations: These often produce no unique peptides because the isoform is just a shorter version of the original.

Implementation: The "Best Candidate" Selection
Instead of just filtering, we will rank peptides for every isoform. This ensures that if a unique peptide exists, you grab it; if not, you flag that isoform for a different enzyme.

In [ ]:
# 1. Setup your source data
dig = digestion_full_proteome_filtered
#dig = digestion_m20_full_proteome_filtered

# 2. Create the Global Peptide Map (Key: Sequence, Value: Set of all IDs containing it)
# This is our "Uniqueness Checker"
peptide_to_ids = dig.groupby('seq')['ID'].apply(set).to_dict()

# 3. Define which IDs you are looking for
# We pull these from your iso_canonical_clean table
target_isoform_ids = iso_canonical_mapping_flat['Isoform_ID'].unique()

# 4. Filter the 'dig' df to only include your target isoforms for the loop
# This replaces 'isoform_to_peptides'
isoform_to_peptides = dig[dig['ID'].isin(target_isoform_ids)].groupby('ID')['seq'].apply(list).to_dict()

final_results = []

# 5. The loop remains mostly the same, now running on target IDs
for iso_id in target_isoform_ids:
    # Get peptides for this ID (if ID exists in digestion)
    peptides = isoform_to_peptides.get(iso_id, [])
    
    unique_to_this_isoform = []
    
    for p in peptides:
        # Check if the set of IDs for this peptide contains ONLY our current iso_id
        if peptide_to_ids.get(p) == {iso_id}:
            unique_to_this_isoform.append(p)
            
    if unique_to_this_isoform:
        # Select longest for MS confidence
        best_peptide = max(unique_to_this_isoform, key=len)
        status = "Unique Found"
    else:
        best_peptide = None
        # Distinguish between 'no peptides at all' and 'all peptides are shared'
        status = "No Unique Peptide (Shared/Truncated)" if peptides else "ID not found in digestion"
        
    final_results.append({
        "Isoform_ID": iso_id,
        "Unique_Peptide": best_peptide,
        "Status": status
    })

results_df = pd.DataFrame(final_results)

In [ ]:
results_df

In [ ]:
print(results_df['Status'].value_counts())

In [ ]:
# 1. Filter for the IDs that were missing entirely from the digestion
missing_from_digestion = results_df[results_df['Status'] == "ID not found in digestion"]['Isoform_ID'].tolist()

print(f"--- Summary Statistics ---")
print(results_df['Status'].value_counts())
print("-" * 30)

if missing_from_digestion:
    print(f" {len(missing_from_digestion)} IDs were NOT found in the digestion dataframe:")
    print(missing_from_digestion)
else:
    print("All target IDs were present in the digestion dataframe.")

Highly specific trypsin:
Unique Found                            14665,
No Unique Peptide (Shared/Truncated)     7405,
ID not found in digestion                 148

20% miscleavage:
Unique Found                            21869,
No Unique Peptide (Shared/Truncated)      201,
ID not found in digestion                 148

Reasons why a protein culd not be found in digestion df: Lack of Cleavage Sites (the protein sequence simply doesn't contain the amino acids the enzyme looks for), Filtering (length filter removed all fragments for that specific protein) or Naming Mismatch.

IDs not matched in ID mapping from UniProt (50 proteins): Q14114-5
Q93009-2
Q13243-4
P30307-5
O00712-3
Q08493-4
O14628-2
O14628-3
Q9HBH9-3
Q08493-6
Q08493-5
P47756-3
Q96LB4-2
Q96JB1-4
Q9NPG1-2
Q8IWY9-3
Q08648-6
Q9BYG7-3
P40855-2
P40855-3
O15409-3
P26367-3
P42702-2
Q13621-2
P52272-3
Q9HBH9-4
O15534-3
Q9UL45-3
Q99259-2
Q9Y281-2
P52757-2
Q9HBH9-5
P10826-4
O15534-2
O00291-2
Q9HAT1-2
Q2M385-2
Q96JB1-3
P13497-7
Q9H0M0-4
Q8WVD5-2
P20963-2
Q9H0M0-5
P40855-4
P04150-4
Q13705-2
O15119-4
Q9BYG7-4
Q08493-7
Q8N7M0-2

For all of these IDs the seqeunce is not available on UniProt.

The other 98 IDs could be mapped but for all of these there are Isoforms with this info: The sequence of this isoform can be found in the external entry linked below. Isoforms of the same protein are often annotated in two different entries if their sequences differ significantly. And this is the one that is linked as the canonical one so this is probably the one that is in the digestion df (I did not check this).

In [ ]:
# 1. Your list of IDs from the prompt
ids_to_check = [
    'P0DPB3-1', 'Q1A5X6-1', 'Q9NPD5-1', 'G3V0H7-1', 'Q9NQG6-1', 'Q5TFQ8-1', 'O00291-2', 'O00712-3', 
    'O14628-2', 'O14628-3', 'O15119-4', 'O15409-3', 'O15534-2', 'O15534-3', 'Q9P0M2-1', 'Q8IX05-1', 
    'B3EWF7-1', 'E9PQ53-1', 'Q5JWF2-1', 'P63092-1', 'P06881-1', 'F8WCM5-1', 'P04150-4', 'F7VJQ1-1', 
    'P01258-1', 'Q6EEV4-1', 'Q9BZG1-1', 'P0DMS9-1', 'P0DMS8-1', 'P35638-1', 'P10826-4', 'P13497-7', 
    'P20963-2', 'P26367-3', 'P30307-5', 'B1AH88-1', 'P0DPQ6-1', 'Q13948-1', 'P40855-2', 'P40855-3', 
    'P40855-4', 'P42167-1', 'P42166-1', 'P42702-2', 'Q8N726-1', 'P47756-3', 'P52272-3', 'P52757-2', 
    'Q9ULB1-1', 'Q9P2S2-1', 'Q6ZVN7-1', 'P63126-1', 'P0DW28-1', 'P0DP91-1', 'Q08493-4', 'Q08493-5', 
    'Q08493-6', 'Q08493-7', 'Q08648-6', 'Q13243-4', 'Q13621-2', 'Q13705-2', 'E9PAV3-1', 'P39880-1', 
    'Q14114-5', 'Q2M385-2', 'O95467-1', 'B7ZAP0-1', 'G9CGD6-1', 'Q9Y3L3-1', 'Q96GD0-1', 'Q70YC5-1', 
    'Q70YC4-1', 'Q8IWY9-3', 'O60449-1', 'Q5JU69-1', 'P42771-1', 'Q9H496-1', 'Q8WVD5-2', 'O95411-1', 
    'Q93009-2', 'L0R6Q1-1', 'Q6ZT62-1', 'Q96JB1-3', 'Q96JB1-4', 'Q96LB4-2', 'Q9BXH1-1', 'Q99259-2', 
    'Q96PG8-2', 'P0DI83-1', 'Q9H0M0-4', 'Q9H0M0-5', 'Q9HBH9-3', 'Q9HBH9-4', 'Q9HBH9-5', 'Q9Y4C0-1', 
    'F5H094-1', 'Q9NPG1-2', 'L0R8F8-1', 'Q9NZJ9-2', 'O43687-1', 'P58401-1', 'Q96EF9-1', 'Q9UL45-3', 
    'P58400-2', 'O94854-3', 'Q9Y281-2', 'Q9HDB5-1', 'O95278-1', 'Q13765-1', 'Q6P9H4-1', 'Q8WWN9-1', 
    'Q9UPN3-1', 'Q92614-1', 'P98175-1', 'P56278-1', 'P56277-1', 'P63128-1', 'Q8N2E6-1', 'O00241-1', 
    'Q9UKY1-1', 'Q96K31-1', 'Q9HAT1-2', 'P0C7T4-1', 'Q5R372-1', 'Q9BQ13-1', 'O95298-1', 'P04156-1', 
    'Q96G79-1', 'Q9NWL6-1', 'A8MTL9-1', 'P0DPB6-1', 'P0CAP2-1', 'Q8N7M0-2', 'Q9HC47-1', 'Q9BYG7-3', 
    'Q9BYG7-4', 'L0R819-1', 'P30536-1', 'P01308-1', 'B3KU38-1', 'Q8IU53-1', 'P60896-1', 'Q6XLA1-1', 
    'Q8NFQ8-1', 'Q96RT6-1', 'Q96PT3-2', 'P0DPK4-2'
]

# 2. List of the 50 IDs with no sequence in UniProt
no_uniprot_seq = {
    'Q14114-5', 'Q93009-2', 'Q13243-4', 'P30307-5', 'O00712-3', 'Q08493-4', 'O14628-2', 'O14628-3', 
    'Q9HBH9-3', 'Q08493-6', 'Q08493-5', 'P47756-3', 'Q96LB4-2', 'Q96JB1-4', 'Q9NPG1-2', 'Q8IWY9-3', 
    'Q08648-6', 'Q9BYG7-3', 'P40855-2', 'P40855-3', 'O15409-3', 'P26367-3', 'P42702-2', 'Q13621-2', 
    'P52272-3', 'Q9HBH9-4', 'O15534-3', 'Q9UL45-3', 'Q99259-2', 'Q9Y281-2', 'P52757-2', 'Q9HBH9-5', 
    'P10826-4', 'O15534-2', 'O00291-2', 'Q9HAT1-2', 'Q2M385-2', 'Q96JB1-3', 'P13497-7', 'Q9H0M0-4', 
    'Q8WVD5-2', 'P20963-2', 'Q9H0M0-5', 'P40855-4', 'P04150-4', 'Q13705-2', 'O15119-4', 'Q9BYG7-4', 
    'Q08493-7', 'Q8N7M0-2'
}

# 3. Get the list of IDs that are marked as 'Canonical' in your mapping df
# We use the 'Canonical_Isoform_ID' column for this check
canonical_set = set(iso_canonical_mapping_flat['Canonical_Isoform_ID'].unique())

# 4. Create the final DataFrame
isoform_check_data = []

for iso_id in ids_to_check:
    isoform_check_data.append({
        "Isoform_ID": iso_id,
        "Is_Canonical": iso_id in canonical_set,
        "Has_UniProt_Sequence": iso_id not in no_uniprot_seq
    })

isoform_check_df = pd.DataFrame(isoform_check_data)

# Display a summary
print(isoform_check_df.head())
print("-" * 30)
print(f"Total Canonical IDs: {isoform_check_df['Is_Canonical'].sum()}")
print(f"Total Missing Sequences: {(~isoform_check_df['Has_UniProt_Sequence']).sum()}")

In [ ]:
print(isoform_check_df[(isoform_check_df["Is_Canonical"] == False) & (isoform_check_df["Has_UniProt_Sequence"] == True)])

Q9NZJ9-2 -> mapped to A0A024RBG1_Q9NZJ9
Q96PT3-2 -> mapped to O43812_Q96PT3 
P0DPK4-2 -> mapped to Q7Z3S9_P0DPK4

all remaining ones are mapped to more than one canonical form but never itself listed as a canonical form (also they are listed as each others isoform in uniprot which to me does not make sense).